In [3]:
"""
aorta and asthma dataset 에서 var 값은 gene symbol이다. 
또한 
ESM3_embeddings.pkl, GPT_embeddings.pkl, Node2vec_embeddings.pkl 의 key 값도 gene symbol 이다.
겹치는 gene symbol list를 1 column tsv 파일로 저장하고,
겹치는 gene symbol list로 aorta, asthma dataset을 필터링하여 저장한다
또

"""
aorta_path = "./aorta/sample_aorta_data_mapped.h5ad"
asthma_path = "./asthma/asthma_data_mapped.h5ad"
esm3_path = "./gene_embedding/ESM3_embeddings.pkl"
genept_path = "./gene_embedding/GPT_embeddings.pkl"

import pickle
import pandas as pd
import scanpy as sc


def load_esm3_embeddings(esm3_path):
    with open(esm3_path, 'rb') as f:
        esm3_embeddings = pickle.load(f)
    return esm3_embeddings
def load_genept_embeddings(genept_path):
    with open(genept_path, 'rb') as f:
        genept_embeddings = pickle.load(f)
    return genept_embeddings
def filter_genes(adata, valid_genes):
    genes_to_keep = [gene for gene in adata.var_names if gene in valid_genes]
    adata_filtered = adata[:, genes_to_keep].copy()
    return adata_filtered


esm3_embeddings = load_esm3_embeddings(esm3_path)
genept_embeddings = load_genept_embeddings(genept_path)
print(f"ESM3 gene count: {len(esm3_embeddings)}")
print(f"GenePT gene count: {len(genept_embeddings)}")
valid_genes = set(esm3_embeddings.keys()).intersection(set(genept_embeddings.keys()))
print(f"Common gene count between ESM3 and GenePT: {len(valid_genes)}")

aorta_adata = sc.read_h5ad(aorta_path)
asthma_adata = sc.read_h5ad(asthma_path)

aorta_filtered = filter_genes(aorta_adata, valid_genes)
asthma_filtered = filter_genes(asthma_adata, valid_genes)
print(f"Aorta filtered gene count: {aorta_filtered.n_vars}")
print(f"Asthma filtered gene count: {asthma_filtered.n_vars}")
# Save the filtered gene list
valid_genes_df = pd.DataFrame(list(valid_genes), columns=["gene_symbol"])
valid_genes_df.to_csv("./common_genes.tsv", sep="\t", index=False, header=False)
# Save the filtered datasets
aorta_filtered.write_h5ad("./aorta/sample_aorta_data_filtered.h5ad")
asthma_filtered.write_h5ad("./asthma/asthma_data_filtered.h5ad")

ESM3 gene count: 15377
GenePT gene count: 133736
Common gene count between ESM3 and GenePT: 15226


/data2/project/bin_jip/miniconda3/envs/biomarker/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/data2/project/bin_jip/miniconda3/envs/biomarker/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Aorta filtered gene count: 13405
Asthma filtered gene count: 14314
